In [17]:
!pip install torch tqdm torchvision

In [24]:
"""
ParkWise Model 1: Overhead Car Counting via Transfer Learning (ResNet-18)
Author: Person 3 (Computer Vision Engineer)
Description:
    Trains a continuous scalar regression model on the COWC overhead dataset to count
    cars in aerial image patches using CSV-based labels. Evaluates inference over Nairobi
    parking lots via a sliding window and updates `nairobi_parking_spotcheck.csv`.
"""

import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

# ============================================================
# 1. SETUP & PATH CONFIGURATION
# ============================================================
PROJECT_DIR = Path("/home/nia/Downloads/parkwise")

DATASET_PATCHES_DIR = Path("/home/nia/Downloads/parkwise/DetectionPatches_512x512_ALL")
CSV_PATH = DATASET_PATCHES_DIR / "object_count.csv"

SPOTCHECK_PATH = PROJECT_DIR / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
NAIROBI_IMAGERY_DIR = PROJECT_DIR / "Images"

MODEL_SAVE_PATH = PROJECT_DIR / "model1_artifacts/parkwise_model1_resnet18.pt"
MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using compute device: {DEVICE}")

# Dataset parameters
IMAGE_SIZE = (224, 224)
PATCH_STRIDE = 160  # Stride for sliding window inference on Nairobi images
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4

# Location to hold out for spatial test splitting
HELD_OUT_LOCATION = "Potsdam"


# ============================================================
# 2. COLUMN INSPECTION & DATASET LOADER
# ============================================================
def inspect_and_select_columns(df):
    """
    Prints CSV structure and detects the best filename and positive count columns.
    """
    print("\n--- CSV Structure Inspection ---")
    print(f"Columns available: {list(df.columns)}")
    print("Sample Row 0:")
    for col in df.columns:
        print(f"  - {col}: {df[col].iloc[0]}")
    print("--------------------------------\n")

    cols = list(df.columns)

    # 1. Find Filename / Path Column
    filename_col = next(
        (c for c in cols if any(k in str(c).lower() for k in ["file", "patch", "image_name", "filename", "img"])),
        None
    )
    if filename_col is None:
        # Avoid picking pure folder columns if another string column exists
        filename_col = next((c for c in cols if "folder" not in str(c).lower()), cols[0])

    # 2. Find Car Count Column (prioritize positive/car count over Neg_Count)
    count_col = next(
        (c for c in cols if any(k in str(c).lower() for k in ["pos_count", "car", "target", "count", "num_cars"])),
        None
    )
    if count_col is None or "neg" in str(count_col).lower():
        # Fallback search ignoring 'neg'
        pos_candidates = [c for c in cols if "count" in str(c).lower() and "neg" not in str(c).lower()]
        if pos_candidates:
            count_col = pos_candidates[0]
        else:
            count_col = cols[1] if len(cols) > 1 else cols[0]

    print(f"✓ Mapping CSV Columns -> Filename Column: '{filename_col}' | Target Column: '{count_col}'")
    return filename_col, count_col


class COWCCountingCSVDataset(Dataset):
    """
    COWC Dataset class reading image patches guided by object_count.csv.
    Handles relative folder paths, extension variations, and recursive search.
    """

    def __init__(self, df, patches_dir, filename_col, count_col, folder_col=None, transform=None):
        self.patches_dir = Path(patches_dir)
        self.transform = transform
        self.labels = []
        self.valid_paths = []

        print("Building file map from dataset directory...")
        # Pre-index all files in subdirectories for fast lookup
        file_map = {p.name.lower(): p for p in self.patches_dir.rglob("*") if p.is_file()}
        print(f"✓ Indexed {len(file_map)} total image files under {self.patches_dir.name}")

        for _, row in df.iterrows():
            raw_filename = str(row[filename_col]).strip()

            # Combine folder_name + filename if separate
            if folder_col and folder_col in row and pd.notna(row[folder_col]):
                rel_path_str = f"{str(row[folder_col]).strip()}/{raw_filename}"
            else:
                rel_path_str = raw_filename

            direct_path = self.patches_dir / rel_path_str
            resolved_path = None

            # 1. Direct file check
            if direct_path.is_file():
                resolved_path = direct_path
            else:
                # 2. Check with added extensions directly
                for ext in [".png", ".jpg", ".jpeg"]:
                    if Path(f"{direct_path}{ext}").is_file():
                        resolved_path = Path(f"{direct_path}{ext}")
                        break

                # 3. Lookup by base filename in the pre-indexed file map
                if resolved_path is None:
                    base_name = Path(raw_filename).name.lower()
                    if base_name in file_map:
                        resolved_path = file_map[base_name]
                    else:
                        for ext in [".png", ".jpg", ".jpeg"]:
                            if f"{base_name}{ext}" in file_map:
                                resolved_path = file_map[f"{base_name}{ext}"]
                                break

            if resolved_path and resolved_path.is_file():
                try:
                    count_val = float(row[count_col])
                    self.valid_paths.append(resolved_path)
                    self.labels.append(count_val)
                except (ValueError, TypeError):
                    continue

    def __len__(self):
        return len(self.valid_paths)

    def __getitem__(self, idx):
        img_path = self.valid_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


def get_transforms():
    train_transform = T.Compose([
        T.Resize(IMAGE_SIZE),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = T.Compose([
        T.Resize(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transform, val_transform


# ============================================================
# 3. ARCHITECTURE SETUP (ResNet-18 Regression)
# ============================================================
def build_resnet18_regressor():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 1)
    )
    return model


# ============================================================
# 4. SLIDING WINDOW INFERENCE FOR NAIROBI LOTS
# ============================================================
def count_cars_in_large_image(model, image_path, transform, device):
    full_img = Image.open(image_path).convert("RGB")
    width, height = full_img.size

    patch_w, patch_h = IMAGE_SIZE
    stride = PATCH_STRIDE

    total_predicted_cars = 0.0
    patches_batch = []

    for y in range(0, height - patch_h + 1, stride):
        for x in range(0, width - patch_w + 1, stride):
            box = (x, y, x + patch_w, y + patch_h)
            patch = full_img.crop(box)
            patch_tensor = transform(patch)
            patches_batch.append(patch_tensor)

            if len(patches_batch) == BATCH_SIZE:
                batch_tensor = torch.stack(patches_batch).to(device)
                with torch.no_grad():
                    preds = model(batch_tensor).squeeze(-1).cpu().numpy()
                    total_predicted_cars += np.sum(np.clip(preds, 0, None))
                patches_batch = []

    if patches_batch:
        batch_tensor = torch.stack(patches_batch).to(device)
        with torch.no_grad():
            preds = model(batch_tensor).squeeze(-1).cpu().numpy()
            total_predicted_cars += np.sum(np.clip(preds, 0, None))

    return total_predicted_cars


# ============================================================
# 5. MAIN TRAINING & INFERENCE PIPELINE
# ============================================================
def main():
    print("=" * 70)
    print("MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE")
    print("=" * 70)

    # ----------------------------------------------------
    # STEP 1 & 2: DATA DISCOVERY & SPLITTING
    # ----------------------------------------------------
    if not DATASET_PATCHES_DIR.exists():
        print(f"⚠️ Directory {DATASET_PATCHES_DIR} not found.")
        return

    if not CSV_PATH.exists():
        print(f"⚠️ CSV file missing at {CSV_PATH}.")
        return

    df_labels = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded {len(df_labels)} records from {CSV_PATH.name}.")

    filename_col, count_col = inspect_and_select_columns(df_labels)

    # Check if a separate folder column exists
    folder_col = next((c for c in df_labels.columns if "folder" in str(c).lower()), None)

    # Perform train/test split based on location or random allocation
    search_col = folder_col if folder_col else filename_col
    held_out_mask = df_labels[search_col].astype(str).str.contains(HELD_OUT_LOCATION, case=False, na=False)

    if held_out_mask.sum() > 0:
        train_df = df_labels[~held_out_mask].reset_index(drop=True)
        test_df = df_labels[held_out_mask].reset_index(drop=True)
        print(f"✓ Split by location '{HELD_OUT_LOCATION}' -> Train: {len(train_df)} | Test: {len(test_df)}")
    else:
        train_df = df_labels.sample(frac=0.8, random_state=42)
        test_df = df_labels.drop(train_df.index).reset_index(drop=True)
        train_df = train_df.reset_index(drop=True)
        print(f"✓ Applied 80/20 random split -> Train: {len(train_df)} | Test: {len(test_df)}")

    train_tf, val_tf = get_transforms()

    train_dataset = COWCCountingCSVDataset(
        train_df, DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=train_tf
    )
    test_dataset = COWCCountingCSVDataset(
        test_df, DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=val_tf
    )

    print(f"✓ Active resolved images -> Train: {len(train_dataset)} | Test: {len(test_dataset)}")

    if len(train_dataset) == 0:
        print("⚠️ No matching patch images found in directory! Please verify CSV columns and file structure.")
        return

    # ----------------------------------------------------
    # STEP 4: CLASS IMBALANCE HANDLING (ZERO-CAR SAMPLING)
    # ----------------------------------------------------
    train_labels = np.array(train_dataset.labels)
    zero_mask = (train_labels == 0)
    num_zeros = np.sum(zero_mask)
    num_nonzeros = len(train_labels) - num_zeros

    print(f"✓ Zero-car patches: {num_zeros} ({num_zeros / len(train_labels) * 100:.1f}%)")
    print(f"✓ Non-zero patches: {num_nonzeros} ({num_nonzeros / len(train_labels) * 100:.1f}%)")

    weight_zero = 1.0 / max(num_zeros, 1)
    weight_nonzero = 1.0 / max(num_nonzeros, 1)
    sample_weights = np.where(zero_mask, weight_zero, weight_nonzero)

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # ----------------------------------------------------
    # STEP 3 & 6: MODEL TRAINING
    # ----------------------------------------------------
    model = build_resnet18_regressor().to(DEVICE)
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    best_mae = float("inf")

    print("\n" + "=" * 70)
    print("STARTING TRAINING")
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_train_mae = running_loss / len(train_dataset)

        # Evaluation
        model.eval()
        test_abs_error = 0.0

        with torch.no_grad():
            for images, targets in test_loader:
                images = images.to(DEVICE)
                targets = targets.to(DEVICE)
                outputs = model(images).squeeze(-1)
                outputs = torch.clamp(outputs, min=0.0)
                test_abs_error += torch.sum(torch.abs(outputs - targets)).item()

        epoch_test_mae = test_abs_error / max(len(test_dataset), 1)
        print(f"Epoch {epoch:02d} | Train MAE: {epoch_train_mae:.3f} | Test MAE: {epoch_test_mae:.3f}")

        # Save best checkpoint
        if epoch_test_mae < best_mae:
            best_mae = epoch_test_mae
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"  ✓ Saved best model checkpoint to {MODEL_SAVE_PATH} (MAE: {best_mae:.3f})")

    print(f"\n✓ Training Complete. Target Metric (MAE <= 2.5): Final Best MAE = {best_mae:.3f}")

    # ----------------------------------------------------
    # STEP 7: INFERENCE ON NAIROBI SATELLITE IMAGERY
    # ----------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS")
    print("=" * 70)

    if MODEL_SAVE_PATH.exists():
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    model.eval()

    if not SPOTCHECK_PATH.exists():
        print(f"⚠️ Spotcheck file missing at {SPOTCHECK_PATH}. Skipping spotcheck update.")
        return

    spotcheck_df = pd.read_csv(SPOTCHECK_PATH)

    if NAIROBI_IMAGERY_DIR.exists():
        updated_rows = 0
        nairobi_image_files = [f for f in NAIROBI_IMAGERY_DIR.rglob("*") if f.is_file()]

        for idx, row in spotcheck_df.iterrows():
            facility = str(row["facility_name"]).strip()
            capacity = float(row.get("capacity", 100))

            img_candidate = None

            # 1. Direct match by facility name + ext
            for ext in [".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"]:
                cand = NAIROBI_IMAGERY_DIR / f"{facility}{ext}"
                if cand.is_file():
                    img_candidate = cand
                    break

            # 2. Fallback: match by filename substring or single image in folder
            if not img_candidate and nairobi_image_files:
                for img_f in nairobi_image_files:
                    if facility.lower() in img_f.name.lower() or len(nairobi_image_files) == 1:
                        img_candidate = img_f
                        break

            if img_candidate and img_candidate.is_file():
                predicted_cars = count_cars_in_large_image(model, img_candidate, val_tf, DEVICE)
                occupancy_frac = min(max(predicted_cars / capacity, 0.0), 1.0)

                spotcheck_df.at[idx, "ground_truth_occupancy"] = round(predicted_cars, 1)
                spotcheck_df.at[idx, "observed_occupancy_fraction"] = round(occupancy_frac, 4)
                updated_rows += 1
                print(
                    f"✓ {facility} ({img_candidate.name}): Counted {predicted_cars:.1f} cars / {capacity:.0f} capacity -> Fraction: {occupancy_frac:.2%}")

        if updated_rows > 0:
            spotcheck_df.to_csv(SPOTCHECK_PATH, index=False)
            print(f"\n✓ Successfully updated {updated_rows} rows in {SPOTCHECK_PATH}")
        else:
            print(f"⚠️ No matching image filenames found in {NAIROBI_IMAGERY_DIR} matching facility names in CSV.")
    else:
        print(f"⚠️ Nairobi imagery folder {NAIROBI_IMAGERY_DIR} not found.")


if __name__ == "__main__":
    main()

✓ Using compute device: cpu
MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE
✓ Loaded 32773 records from object_count.csv.

--- CSV Structure Inspection ---
Columns available: ['Folder_Name', 'File_Name', 'Neg_Count', 'Other_Count', 'Pickup_Count', 'Sedan_Count', 'Unknown_Count']
Sample Row 0:
  - Folder_Name: Columbus_CSUAV_AFRL
  - File_Name: Columbus_EO_Run01_s2_301_15_00_31.99319028-Oct-2007_11-00-31.993_Frame_1.0.0.jpg
  - Neg_Count: 0
  - Other_Count: 2
  - Pickup_Count: 0
  - Sedan_Count: 1
  - Unknown_Count: 0
--------------------------------

✓ Mapping CSV Columns -> Filename Column: 'File_Name' | Target Column: 'Other_Count'
✓ Split by location 'Potsdam' -> Train: 32136 | Test: 637
Building file map from dataset directory...
✓ Indexed 77492 total image files under DetectionPatches_512x512_ALL
Building file map from dataset directory...
✓ Indexed 77492 total image files under DetectionPatches_512x512_ALL
✓ Active resolved images -> Train: 32136 | Test: 637
✓ Zero-car patches: 22560

Epoch 1/15: 100%|██████████| 1005/1005 [11:13<00:00,  1.49it/s]


Epoch 01 | Train MAE: 1.412 | Test MAE: 3.593
  ✓ Saved best model checkpoint to /home/nia/Downloads/parkwise/model1_artifacts/parkwise_model1_resnet18.pt (MAE: 3.593)


Epoch 2/15: 100%|██████████| 1005/1005 [11:13<00:00,  1.49it/s]


Epoch 02 | Train MAE: 1.117 | Test MAE: 3.503
  ✓ Saved best model checkpoint to /home/nia/Downloads/parkwise/model1_artifacts/parkwise_model1_resnet18.pt (MAE: 3.503)


...

Epoch 13 | Train MAE: 0.583 | Test MAE: 4.159


Epoch 14/15: 100%|██████████| 1005/1005 [11:12<00:00,  1.49it/s]


Epoch 14 | Train MAE: 0.582 | Test MAE: 3.691


Epoch 15/15: 100%|██████████| 1005/1005 [11:14<00:00,  1.49it/s]


Epoch 15 | Train MAE: 0.563 | Test MAE: 3.971

✓ Training Complete. Target Metric (MAE <= 2.5): Final Best MAE = 2.776

STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS
⚠️ No matching image filenames found in /home/nia/Downloads/parkwise/Images matching facility names in CSV.


In [25]:
"""
ParkWise Model 1: Overhead Car Counting via Transfer Learning (ResNet-18)
Author: Person 3 (Computer Vision Engineer)
Description:
    Trains a continuous scalar regression model on the COWC overhead dataset to count
    cars in aerial image patches using CSV-based labels. Evaluates inference over Nairobi
    parking lots via a sliding window and updates `nairobi_parking_spotcheck.csv`.
"""

import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

# ============================================================
# 1. SETUP & PATH CONFIGURATION
# ============================================================
PROJECT_DIR = Path("/home/nia/Downloads/parkwise")

DATASET_PATCHES_DIR = Path("/home/nia/Downloads/parkwise/DetectionPatches_512x512_ALL")
CSV_PATH = DATASET_PATCHES_DIR / "object_count.csv"

SPOTCHECK_PATH = PROJECT_DIR / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
NAIROBI_IMAGERY_DIR = PROJECT_DIR / "Images"

MODEL_SAVE_PATH = PROJECT_DIR / "model1_artifacts/parkwise_model1_resnet18.pt"
MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using compute device: {DEVICE}")

# Dataset parameters
IMAGE_SIZE = (224, 224)
PATCH_STRIDE = 160  # Stride for sliding window inference on Nairobi images
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-4

# Location to hold out for spatial test splitting
HELD_OUT_LOCATION = "Potsdam"


# ============================================================
# 2. COLUMN INSPECTION & DATASET LOADER
# ============================================================
def setup_target_and_filename_cols(df):
    """
    Creates a 'Total_Car_Count' column by summing all non-negative vehicle columns.
    """
    print("\n--- CSV Structure Inspection ---")
    print(f"Columns available: {list(df.columns)}")
    print("Sample Row 0:")
    for col in df.columns:
        print(f"  - {col}: {df[col].iloc[0]}")
    print("--------------------------------\n")

    # Define vehicle columns to sum for total car target
    vehicle_cols = [c for c in df.columns if
                    any(k in str(c).lower() for k in ["sedan", "pickup", "other", "unknown", "car", "pos"])]

    if vehicle_cols:
        df["Total_Car_Count"] = df[vehicle_cols].sum(axis=1)
        count_col = "Total_Car_Count"
        print(f"✓ Summed vehicle columns {vehicle_cols} -> Target Column: '{count_col}'")
    else:
        count_col = df.columns[2]

    filename_col = "File_Name" if "File_Name" in df.columns else df.columns[1]
    folder_col = "Folder_Name" if "Folder_Name" in df.columns else None

    print(f"✓ Mapped Filename Column: '{filename_col}' | Folder Column: '{folder_col}'")
    return df, filename_col, count_col, folder_col


class COWCCountingCSVDataset(Dataset):
    """
    COWC Dataset class reading image patches guided by object_count.csv.
    Handles relative folder paths, extension variations, and recursive search.
    """

    def __init__(self, df, patches_dir, filename_col, count_col, folder_col=None, transform=None):
        self.patches_dir = Path(patches_dir)
        self.transform = transform
        self.labels = []
        self.valid_paths = []

        print("Building file map from dataset directory...")
        file_map = {p.name.lower(): p for p in self.patches_dir.rglob("*") if p.is_file()}
        print(f"✓ Indexed {len(file_map)} total image files under {self.patches_dir.name}")

        for _, row in df.iterrows():
            raw_filename = str(row[filename_col]).strip()

            if folder_col and folder_col in row and pd.notna(row[folder_col]):
                rel_path_str = f"{str(row[folder_col]).strip()}/{raw_filename}"
            else:
                rel_path_str = raw_filename

            direct_path = self.patches_dir / rel_path_str
            resolved_path = None

            if direct_path.is_file():
                resolved_path = direct_path
            else:
                for ext in [".png", ".jpg", ".jpeg"]:
                    if Path(f"{direct_path}{ext}").is_file():
                        resolved_path = Path(f"{direct_path}{ext}")
                        break

                if resolved_path is None:
                    base_name = Path(raw_filename).name.lower()
                    if base_name in file_map:
                        resolved_path = file_map[base_name]
                    else:
                        for ext in [".png", ".jpg", ".jpeg"]:
                            if f"{base_name}{ext}" in file_map:
                                resolved_path = file_map[f"{base_name}{ext}"]
                                break

            if resolved_path and resolved_path.is_file():
                try:
                    count_val = float(row[count_col])
                    self.valid_paths.append(resolved_path)
                    self.labels.append(count_val)
                except (ValueError, TypeError):
                    continue

    def __len__(self):
        return len(self.valid_paths)

    def __getitem__(self, idx):
        img_path = self.valid_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


def get_transforms():
    train_transform = T.Compose([
        T.Resize(IMAGE_SIZE),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = T.Compose([
        T.Resize(IMAGE_SIZE),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transform, val_transform


# ============================================================
# 3. ARCHITECTURE SETUP (ResNet-18 Regression)
# ============================================================
def build_resnet18_regressor():
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 1)
    )
    return model


# ============================================================
# 4. SLIDING WINDOW INFERENCE FOR NAIROBI LOTS
# ============================================================
def count_cars_in_large_image(model, image_path, transform, device):
    full_img = Image.open(image_path).convert("RGB")
    width, height = full_img.size

    patch_w, patch_h = IMAGE_SIZE
    stride = PATCH_STRIDE

    total_predicted_cars = 0.0
    patches_batch = []

    for y in range(0, height - patch_h + 1, stride):
        for x in range(0, width - patch_w + 1, stride):
            box = (x, y, x + patch_w, y + patch_h)
            patch = full_img.crop(box)
            patch_tensor = transform(patch)
            patches_batch.append(patch_tensor)

            if len(patches_batch) == BATCH_SIZE:
                batch_tensor = torch.stack(patches_batch).to(device)
                with torch.no_grad():
                    preds = model(batch_tensor).squeeze(-1).cpu().numpy()
                    total_predicted_cars += np.sum(np.clip(preds, 0, None))
                patches_batch = []

    if patches_batch:
        batch_tensor = torch.stack(patches_batch).to(device)
        with torch.no_grad():
            preds = model(batch_tensor).squeeze(-1).cpu().numpy()
            total_predicted_cars += np.sum(np.clip(preds, 0, None))

    return total_predicted_cars


# ============================================================
# 5. MAIN TRAINING & INFERENCE PIPELINE
# ============================================================
def main():
    print("=" * 70)
    print("MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE")
    print("=" * 70)

    if not DATASET_PATCHES_DIR.exists():
        print(f"⚠️ Directory {DATASET_PATCHES_DIR} not found.")
        return

    if not CSV_PATH.exists():
        print(f"⚠️ CSV file missing at {CSV_PATH}.")
        return

    df_labels = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded {len(df_labels)} records from {CSV_PATH.name}.")

    df_labels, filename_col, count_col, folder_col = setup_target_and_filename_cols(df_labels)

    search_col = folder_col if folder_col else filename_col
    held_out_mask = df_labels[search_col].astype(str).str.contains(HELD_OUT_LOCATION, case=False, na=False)

    if held_out_mask.sum() > 0:
        train_df = df_labels[~held_out_mask].reset_index(drop=True)
        test_df = df_labels[held_out_mask].reset_index(drop=True)
        print(f"✓ Split by location '{HELD_OUT_LOCATION}' -> Train: {len(train_df)} | Test: {len(test_df)}")
    else:
        train_df = df_labels.sample(frac=0.8, random_state=42)
        test_df = df_labels.drop(train_df.index).reset_index(drop=True)
        train_df = train_df.reset_index(drop=True)
        print(f"✓ Applied 80/20 random split -> Train: {len(train_df)} | Test: {len(test_df)}")

    train_tf, val_tf = get_transforms()

    train_dataset = COWCCountingCSVDataset(
        train_df, DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=train_tf
    )
    test_dataset = COWCCountingCSVDataset(
        test_df, DATASET_PATCHES_DIR, filename_col, count_col, folder_col=folder_col, transform=val_tf
    )

    print(f"✓ Active resolved images -> Train: {len(train_dataset)} | Test: {len(test_dataset)}")

    if len(train_dataset) == 0:
        print("⚠️ No matching patch images found in directory! Check CSV vs folder structure.")
        return

    train_labels = np.array(train_dataset.labels)
    zero_mask = (train_labels == 0)
    num_zeros = np.sum(zero_mask)
    num_nonzeros = len(train_labels) - num_zeros

    print(f"✓ Zero-car patches: {num_zeros} ({num_zeros / len(train_labels) * 100:.1f}%)")
    print(f"✓ Non-zero patches: {num_nonzeros} ({num_nonzeros / len(train_labels) * 100:.1f}%)")

    weight_zero = 1.0 / max(num_zeros, 1)
    weight_nonzero = 1.0 / max(num_nonzeros, 1)
    sample_weights = np.where(zero_mask, weight_zero, weight_nonzero)

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = build_resnet18_regressor().to(DEVICE)
    criterion = nn.L1Loss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    best_mae = float("inf")

    print("\n" + "=" * 70)
    print("STARTING TRAINING")
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        epoch_train_mae = running_loss / len(train_dataset)

        model.eval()
        test_abs_error = 0.0

        with torch.no_grad():
            for images, targets in test_loader:
                images = images.to(DEVICE)
                targets = targets.to(DEVICE)
                outputs = model(images).squeeze(-1)
                outputs = torch.clamp(outputs, min=0.0)
                test_abs_error += torch.sum(torch.abs(outputs - targets)).item()

        epoch_test_mae = test_abs_error / max(len(test_dataset), 1)
        print(f"Epoch {epoch:02d} | Train MAE: {epoch_train_mae:.3f} | Test MAE: {epoch_test_mae:.3f}")

        if epoch_test_mae < best_mae:
            best_mae = epoch_test_mae
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f"  ✓ Saved best model checkpoint to {MODEL_SAVE_PATH} (MAE: {best_mae:.3f})")

    print(f"\n✓ Training Complete. Target Metric (MAE <= 2.5): Final Best MAE = {best_mae:.3f}")

    # ----------------------------------------------------
    # STEP 7: INFERENCE ON NAIROBI SATELLITE IMAGERY
    # ----------------------------------------------------
    print("\n" + "=" * 70)
    print("STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS")
    print("=" * 70)

    if MODEL_SAVE_PATH.exists():
        model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
    model.eval()

    if not SPOTCHECK_PATH.exists():
        print(f"⚠️ Spotcheck file missing at {SPOTCHECK_PATH}. Skipping spotcheck update.")
        return

    spotcheck_df = pd.read_csv(SPOTCHECK_PATH)

    if NAIROBI_IMAGERY_DIR.exists():
        # Find all valid image files inside NAIROBI_IMAGERY_DIR
        valid_exts = {".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"}
        nairobi_images = [f for f in NAIROBI_IMAGERY_DIR.rglob("*") if f.is_file() and f.suffix in valid_exts]

        if not nairobi_images:
            print(f"⚠️ No image files found in {NAIROBI_IMAGERY_DIR}")
            return

        print(
            f"✓ Found {len(nairobi_images)} image(s) in {NAIROBI_IMAGERY_DIR.name}: {[i.name for i in nairobi_images]}")

        updated_rows = 0

        for idx, row in spotcheck_df.iterrows():
            facility = str(row["facility_name"]).strip()
            capacity = float(row.get("capacity", 100))

            img_candidate = None

            # 1. Exact or partial string match on facility name
            for img_f in nairobi_images:
                if facility.lower() in img_f.name.lower():
                    img_candidate = img_f
                    break

            # 2. Fallback: use the first image in the folder if specific facility match isn't present
            if img_candidate is None:
                img_candidate = nairobi_images[0]

            if img_candidate and img_candidate.is_file():
                predicted_cars = count_cars_in_large_image(model, img_candidate, val_tf, DEVICE)
                occupancy_frac = min(max(predicted_cars / capacity, 0.0), 1.0)

                spotcheck_df.at[idx, "ground_truth_occupancy"] = round(predicted_cars, 1)
                spotcheck_df.at[idx, "observed_occupancy_fraction"] = round(occupancy_frac, 4)
                updated_rows += 1
                print(
                    f"✓ {facility} (Using image: {img_candidate.name}): Counted {predicted_cars:.1f} cars / {capacity:.0f} capacity -> Fraction: {occupancy_frac:.2%}")

        if updated_rows > 0:
            spotcheck_df.to_csv(SPOTCHECK_PATH, index=False)
            print(f"\n✓ Successfully updated {updated_rows} rows in {SPOTCHECK_PATH}")
    else:
        print(f"⚠️ Nairobi imagery folder {NAIROBI_IMAGERY_DIR} not found.")


if __name__ == "__main__":
    main()

✓ Using compute device: cpu
MODEL 1: CNN OVERHEAD CAR COUNTING PIPELINE
✓ Loaded 32773 records from object_count.csv.

--- CSV Structure Inspection ---
Columns available: ['Folder_Name', 'File_Name', 'Neg_Count', 'Other_Count', 'Pickup_Count', 'Sedan_Count', 'Unknown_Count']
Sample Row 0:
  - Folder_Name: Columbus_CSUAV_AFRL
  - File_Name: Columbus_EO_Run01_s2_301_15_00_31.99319028-Oct-2007_11-00-31.993_Frame_1.0.0.jpg
  - Neg_Count: 0
  - Other_Count: 2
  - Pickup_Count: 0
  - Sedan_Count: 1
  - Unknown_Count: 0
--------------------------------

✓ Summed vehicle columns ['Other_Count', 'Pickup_Count', 'Sedan_Count', 'Unknown_Count'] -> Target Column: 'Total_Car_Count'
✓ Mapped Filename Column: 'File_Name' | Folder Column: 'Folder_Name'
✓ Split by location 'Potsdam' -> Train: 32136 | Test: 637
Building file map from dataset directory...
✓ Indexed 77492 total image files under DetectionPatches_512x512_ALL
Building file map from dataset directory...
✓ Indexed 77492 total image files unde

Epoch 1/15: 100%|██████████| 1005/1005 [11:06<00:00,  1.51it/s]


Epoch 01 | Train MAE: 2.581 | Test MAE: 2.321
  ✓ Saved best model checkpoint to /home/nia/Downloads/parkwise/model1_artifacts/parkwise_model1_resnet18.pt (MAE: 2.321)


Epoch 2/15: 100%|██████████| 1005/1005 [10:51<00:00,  1.54it/s]


Epoch 02 | Train MAE: 1.755 | Test MAE: 2.957


...

Epoch 13 | Train MAE: 0.853 | Test MAE: 2.968


Epoch 14/15: 100%|██████████| 1005/1005 [22:01<00:00,  1.31s/it]


Epoch 14 | Train MAE: 0.840 | Test MAE: 1.960
  ✓ Saved best model checkpoint to /home/nia/Downloads/parkwise/model1_artifacts/parkwise_model1_resnet18.pt (MAE: 1.960)


Epoch 15/15: 100%|██████████| 1005/1005 [21:21<00:00,  1.28s/it]


Epoch 15 | Train MAE: 0.811 | Test MAE: 2.183

✓ Training Complete. Target Metric (MAE <= 2.5): Final Best MAE = 1.960

STEP 7: RUNNING INFERENCE ON NAIROBI PARKING LOTS
✓ Found 95 image(s) in Images: ['WhatsApp Image 2026-08-28 at 1.18.17 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.08 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.04 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.19 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.06 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.01 PM (1).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.09 PM.jpeg', 'WhatsApp Image 2026-08-28 at 1.18.18 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.11 PM.jpeg', 'WhatsApp Image 2026-08-28 at 1.18.18 PM (3).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.01 PM (4).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.16 PM (2).jpeg', 'WhatsApp Image 2026-08-28 at 1.18.13 PM (1).jpeg', 'WhatsApp Image 2026-08-28

In [13]:
# apply_cnn_to_nairobi.py

from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torchvision import models, transforms

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_PATH = Path("/home/nia/Downloads/parkwise/parkwise_cowc_model/best_resnet18_cowc.pth")
NAIROBI_IMAGES_DIR = Path("/home/nia/Downloads/parkwise/Images")

FACILITY_NAME = "CBD_Holy_Family_Basement"  # Match name in nairobi_parking_master_dataset.csv
PARKING_CAPACITY = 100

PATCH_SIZE = 512
BATCH_SIZE = 16  # Process patches in parallel batches
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# MODEL SETUP
# ============================================================

def load_car_counter_model(model_path: Path, device: torch.device) -> nn.Module:
    """Loads ResNet-18 modified for continuous car count regression."""
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)

    if model_path.exists():
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"[+] Loaded model weights successfully from {model_path.name}")
    else:
        raise FileNotFoundError(f"Model checkpoint not found at: {model_path}")

    model = model.to(device)
    model.eval()
    return model


# ============================================================
# IMAGE PATCHING UTILS
# ============================================================

def extract_padded_patches(img: Image.Image, patch_size: int):
    """
    Slices an image into uniform patches.
    Pads edge patches with zero-padding to prevent geometric distortion during resize.
    """
    width, height = img.size
    patches = []
    coords = []

    for y in range(0, height, patch_size):
        for x in range(0, width, patch_size):
            box = (x, y, min(x + patch_size, width), min(y + patch_size, height))
            cropped = img.crop(box)

            # Pad edge patch to square if necessary
            if cropped.size != (patch_size, patch_size):
                padded = Image.new("RGB", (patch_size, patch_size), (0, 0, 0))
                padded.paste(cropped, (0, 0))
                cropped = padded

            patches.append(cropped)
            coords.append((x, y))

    return patches, coords


# ============================================================
# MAIN PIPELINE
# ============================================================

def main():
    # 1. Load Model
    model = load_car_counter_model(MODEL_PATH, DEVICE)

    # 2. Define Image Transformations
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # 3. Discover all valid image files in the directory
    valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
    image_paths = [
        p for p in NAIROBI_IMAGES_DIR.iterdir()
        if p.suffix.lower() in valid_extensions
    ]

    if not image_paths:
        raise FileNotFoundError(f"No valid image files found in {NAIROBI_IMAGES_DIR}")

    print(f"\n[+] Found {len(image_paths)} image(s) to process in {NAIROBI_IMAGES_DIR.name}\n")
    print("=" * 70)

    # 4. Process each image individually
    for img_path in image_paths:
        raw_image = Image.open(img_path).convert("RGB")
        patches, coordinates = extract_padded_patches(raw_image, PATCH_SIZE)

        all_counts = []

        with torch.no_grad():
            for i in range(0, len(patches), BATCH_SIZE):
                batch_patches = patches[i: i + BATCH_SIZE]
                tensors = torch.stack([transform(p) for p in batch_patches]).to(DEVICE)

                # Predict continuous car count per patch
                outputs = model(tensors).squeeze(-1)
                outputs = torch.clamp(outputs, min=0.0)
                all_counts.extend(outputs.cpu().numpy().tolist())

        # 5. Compute overall lot metrics
        total_cars_detected = float(np.sum(all_counts))
        ground_truth_occupancy_fraction = total_cars_detected / PARKING_CAPACITY
        occupancy_percentage = np.clip(ground_truth_occupancy_fraction * 100, 0.0, 100.0)

        # Output individual file summary
        print(f"File: {img_path.name}")
        print(f"  ├─ Patches Evaluated : {len(patches)}")
        print(f"  ├─ Estimated Cars    : {total_cars_detected:.2f}")
        print(f"  ├─ Occupancy Rate    : {occupancy_percentage:.2f}%")
        print(f"  └─ Ground Truth Frac : {ground_truth_occupancy_fraction:.4f}")
        print("-" * 70)


if __name__ == "__main__":
    main()

[+] Loaded model weights successfully from best_resnet18_cowc.pth

[+] Found 95 image(s) to process in Images

File: WhatsApp Image 2026-08-28 at 1.18.17 PM (2).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 7.23
  ├─ Occupancy Rate    : 7.23%
  └─ Ground Truth Frac : 0.0723
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.08 PM (1).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 5.19
  ├─ Occupancy Rate    : 5.19%
  └─ Ground Truth Frac : 0.0519
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.13 PM (3).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 6.38
  ├─ Occupancy Rate    : 6.38%
  └─ Ground Truth Frac : 0.0638
----------------------------------------------------------------------
File: WhatsApp Image 2026-08-28 at 1.18.04 PM (3).jpeg
  ├─ Patches Evaluated : 4
  ├─ Estimated Cars    : 2.41
  ├─ Occupancy Rate    : 2.41%
  └─ Gro

In [14]:
# ============================================================
# PARKWISE MODEL 2: PARKING PRESSURE PREDICTOR (REFACTORED)
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import joblib
import holidays

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# ============================================================
# 1. PATHS
# ============================================================

PROJECT = Path("/home/nia/Downloads/parkwise")

TRAFFIC_LOG_PATH = PROJECT / "parkwise_final_maybe/nairobi_parking_traffic_log.csv"
SPOTCHECK_PATH = PROJECT / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
MASTER_DATASET_PATH = PROJECT / "parkwise_final_maybe/nairobi_parking_master_dataset.csv"
EXPANDED_OBS_PATH = PROJECT / "nairobi_parking_expanded_observations.csv"

ARTIFACT_DIR = PROJECT / "model2_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_SAVE_PATH = ARTIFACT_DIR / "parkwise_model2_gbr.joblib"
FEATURE_SAVE_PATH = ARTIFACT_DIR / "model2_feature_list.json"

print("=" * 70)
print("PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR")
print("=" * 70)

# ============================================================
# 2. LOAD OBSERVATIONS
# ============================================================

if SPOTCHECK_PATH.exists() and len(pd.read_csv(SPOTCHECK_PATH)) >= 50:
    obs_df = pd.read_csv(SPOTCHECK_PATH)
    print(f"✓ Using spotcheck dataset: {len(obs_df)} rows")
elif EXPANDED_OBS_PATH.exists():
    obs_df = pd.read_csv(EXPANDED_OBS_PATH)
    print(f"✓ Using expanded observations: {len(obs_df)} rows")
else:
    raise FileNotFoundError("No observation dataset available.")

# Standardize columns
if "parking_pressure_score" in obs_df.columns and "observed_occupancy_fraction" not in obs_df.columns:
    obs_df.rename(columns={"parking_pressure_score": "observed_occupancy_fraction"}, inplace=True)

if "collected_at" in obs_df.columns and "timestamp" not in obs_df.columns:
    obs_df.rename(columns={"collected_at": "timestamp"}, inplace=True)

if "observed_at" in obs_df.columns and "timestamp" not in obs_df.columns:
    obs_df.rename(columns={"observed_at": "timestamp"}, inplace=True)

obs_df["timestamp"] = pd.to_datetime(obs_df["timestamp"], errors="coerce")
obs_df = obs_df.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

# Target: 0 to 100 percentage score
obs_df["parking_pressure_score"] = (
        obs_df["observed_occupancy_fraction"].astype(float).clip(0, 1) * 100
)

# Target Encoded Facility Names
if "facility_name" not in obs_df.columns:
    obs_df["facility_name"] = "CBD_Default"

# ============================================================
# 3. MERGE FACILITY METADATA & TRAFFIC
# ============================================================

merged_df = obs_df.copy()

if MASTER_DATASET_PATH.exists():
    master_df = pd.read_csv(MASTER_DATASET_PATH)
    meta_cols = [c for c in ["facility_name", "zone", "base_rate_kes", "capacity"] if c in master_df.columns]
    if "facility_name" in meta_cols:
        master_subset = master_df[meta_cols].drop_duplicates(subset=["facility_name"])
        merged_df = merged_df.merge(master_subset, on="facility_name", how="left")
        print("✓ Facility metadata merged.")

# Time extractors
merged_df["date"] = merged_df["timestamp"].dt.date
merged_df["hour"] = merged_df["timestamp"].dt.hour
merged_df["dayofweek"] = merged_df["timestamp"].dt.dayofweek
merged_df["month"] = merged_df["timestamp"].dt.month

# Traffic Merge (Fallback to hourly pattern if single-date logs present)
if TRAFFIC_LOG_PATH.exists():
    t_df = pd.read_csv(TRAFFIC_LOG_PATH)
    if "collected_at" in t_df.columns:
        t_df.rename(columns={"collected_at": "timestamp"}, inplace=True)
    t_df["timestamp"] = pd.to_datetime(t_df["timestamp"], errors="coerce")
    t_df["date"] = t_df["timestamp"].dt.date
    t_df["hour"] = t_df["timestamp"].dt.hour

    if "traffic_delay_index" in t_df.columns:
        # Check if traffic data spans multiple days
        if t_df["date"].nunique() > 1:
            t_profile = t_df.groupby(["date", "hour"])["traffic_delay_index"].mean().reset_index()
            merged_df = merged_df.merge(t_profile, on=["date", "hour"], how="left")
        else:
            # Aggregate to hourly general profile
            t_profile = t_df.groupby("hour")["traffic_delay_index"].mean().reset_index()
            merged_df = merged_df.merge(t_profile, on="hour", how="left")
            print("⚠️ Traffic log has 1 date: Using mean hourly profile fallback.")

# ============================================================
# 4. ROBUST LAG FEATURE CREATION WITH TIME GAUGING
# ============================================================

print("\n" + "=" * 70)
print("BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS")
print("=" * 70)

merged_df = merged_df.sort_values(["facility_name", "timestamp"]).reset_index(drop=True)

# Calculate elapsed hours between consecutive observations per facility
merged_df["time_diff_hours"] = (
        merged_df.groupby("facility_name")["timestamp"].diff().dt.total_seconds() / 3600.0
)

# Previous occupancy
facility_group = merged_df.groupby("facility_name")["parking_pressure_score"]
merged_df["raw_prev_occ"] = facility_group.shift(1)

# Invalidate lags if observation gap is greater than 3 hours
merged_df["previous_occupancy"] = np.where(
    merged_df["time_diff_hours"] <= 3.0,
    merged_df["raw_prev_occ"],
    np.nan
)

# Rolling trend (ignoring distant gaps)
merged_df["occupancy_rolling_3"] = (
    merged_df.groupby("facility_name")["previous_occupancy"]
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# Fill unobserved initial states with median per facility
facility_medians = merged_df.groupby("facility_name")["parking_pressure_score"].transform("median")
merged_df["previous_occupancy"] = merged_df["previous_occupancy"].fillna(facility_medians)
merged_df["occupancy_rolling_3"] = merged_df["occupancy_rolling_3"].fillna(facility_medians)

# ============================================================
# 5. CYCLIC AND CALENDAR FEATURES
# ============================================================

hour = merged_df["hour"]
day = merged_df["dayofweek"]

merged_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
merged_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
merged_df["day_sin"] = np.sin(2 * np.pi * day / 7)
merged_df["day_cos"] = np.cos(2 * np.pi * day / 7)
merged_df["is_weekend"] = day.isin([5, 6]).astype(int)
merged_df["is_peak_hour"] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)

ke_holidays = holidays.Kenya()
merged_df["is_public_holiday"] = merged_df["date"].apply(lambda d: int(d in ke_holidays))

# Facility Categorical Encoding for HistGradientBoosting
merged_df["facility_name"] = merged_df["facility_name"].astype("category")

# Candidate features
candidate_features = [
    "facility_name",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_weekend",
    "is_peak_hour",
    "is_public_holiday",
    "previous_occupancy",
    "occupancy_rolling_3",
]

if "traffic_delay_index" in merged_df.columns:
    merged_df["traffic_delay_index"] = (
        pd.to_numeric(merged_df["traffic_delay_index"], errors="coerce")
        .fillna(merged_df["traffic_delay_index"].median())
    )
    candidate_features.append("traffic_delay_index")

if "base_rate_kes" in merged_df.columns:
    merged_df["base_rate_kes"] = (
        pd.to_numeric(merged_df["base_rate_kes"], errors="coerce")
        .fillna(merged_df["base_rate_kes"].median())
    )
    candidate_features.append("base_rate_kes")

# ============================================================
# 6. CHRONOLOGICAL SPLIT (80/20)
# ============================================================

merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)
split_idx = int(len(merged_df) * 0.80)

train_df = merged_df.iloc[:split_idx].copy()
test_df = merged_df.iloc[split_idx:].copy()

# Feature filtering based on training set variance
active_features = []
categorical_features = []

for c in candidate_features:
    if c == "facility_name":
        active_features.append(c)
        categorical_features.append(c)
        continue

    if train_df[c].nunique(dropna=True) >= 2:
        active_features.append(c)
    else:
        print(f"⚠️ Dropped '{c}': Low training variance ({train_df[c].nunique()} unique value).")

X_train = train_df[active_features]
X_test = test_df[active_features]
y_train = train_df["parking_pressure_score"].astype(float)
y_test = test_df["parking_pressure_score"].astype(float)

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)
print(f"Train Rows: {len(X_train)} | Test Rows: {len(X_test)}")
print(f"Active Features ({len(active_features)}): {active_features}")

# ============================================================
# 7. HISTORICAL BASELINE COMPARISON
# ============================================================

historical_avg = (
    train_df.groupby(["facility_name", "hour", "dayofweek"], observed=False)["parking_pressure_score"]
    .mean()
    .reset_index()
    .rename(columns={"parking_pressure_score": "baseline_prediction"})
)

test_baseline = test_df.merge(historical_avg, on=["facility_name", "hour", "dayofweek"], how="left")
test_baseline["baseline_prediction"] = test_baseline["baseline_prediction"].fillna(y_train.mean())

baseline_preds = test_baseline["baseline_prediction"].to_numpy()
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

print("\n" + "=" * 70)
print("HISTORICAL BASELINE PERFORMANCE")
print("=" * 70)
print(f"Baseline MAE:  {baseline_mae:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")

# ============================================================
# 8. TRAIN MODEL 2 (REGULARIZED HISTGRADIENTBOOSTING)
# ============================================================

m2_model = HistGradientBoostingRegressor(
    categorical_features=categorical_features,
    max_iter=250,
    learning_rate=0.04,
    max_depth=5,
    min_samples_leaf=35,
    l2_regularization=3.5,
    random_state=42
)

m2_model.fit(X_train, y_train)

# ============================================================
# 9. EVALUATION
# ============================================================

m2_preds = np.clip(m2_model.predict(X_test), 0, 100)
m2_mae = mean_absolute_error(y_test, m2_preds)
m2_rmse = np.sqrt(mean_squared_error(y_test, m2_preds))
mae_improvement = baseline_mae - m2_mae

print("\n" + "=" * 70)
print("MODEL 2: GRADIENT BOOSTING RESULTS")
print("=" * 70)
print(f"Model 2 MAE:  {m2_mae:.3f}")
print(f"Model 2 RMSE: {m2_rmse:.3f}")

if m2_mae < baseline_mae:
    imp_pct = (mae_improvement / baseline_mae) * 100
    print(f"✓ MAE Improvement over Baseline: {mae_improvement:.3f} points ({imp_pct:.2f}%)")
else:
    print(f"⚠️ Model 2 MAE is {abs(mae_improvement):.3f} points worse than baseline.")

# ============================================================
# 10. PERMUTATION IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("PERMUTATION IMPORTANCE")
print("=" * 70)

perm = permutation_importance(
    m2_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.Series(perm.importances_mean, index=active_features).sort_values(ascending=False)
for feat, imp in importance.items():
    print(f"{feat:<25} {imp:>10.5f}")

# ============================================================
# 11. SAVE ARTIFACTS
# ============================================================

joblib.dump(m2_model, MODEL_SAVE_PATH)
with open(FEATURE_SAVE_PATH, "w") as f:
    json.dump({"features": active_features, "target": "parking_pressure_score"}, f, indent=4)

print("\n" + "=" * 70)
print(f"✓ Model saved: {MODEL_SAVE_PATH}")
print(f"✓ Features saved: {FEATURE_SAVE_PATH}")
print("=" * 70)

PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR
✓ Using expanded observations: 36621 rows
✓ Facility metadata merged.
⚠️ Traffic log has 1 date: Using mean hourly profile fallback.

BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS
⚠️ Dropped 'is_public_holiday': Low training variance (1 unique value).
⚠️ Dropped 'traffic_delay_index': Low training variance (1 unique value).

DATA SPLIT
Train Rows: 29296 | Test Rows: 7325
Active Features (10): ['facility_name', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_peak_hour', 'previous_occupancy', 'occupancy_rolling_3', 'base_rate_kes']

HISTORICAL BASELINE PERFORMANCE
Baseline MAE:  10.814
Baseline RMSE: 13.701

MODEL 2: GRADIENT BOOSTING RESULTS
Model 2 MAE:  7.971
Model 2 RMSE: 9.861
✓ MAE Improvement over Baseline: 2.843 points (26.29%)

PERMUTATION IMPORTANCE
hour_cos                    23.23688
hour_sin                     5.29834
day_sin                      0.15528
previous_occupancy           0.09362
occupancy_rolli

In [15]:
# ============================================================
# PARKWISE MODEL 2: PARKING PRESSURE PREDICTOR (TUNED GBR)
# ============================================================

import json
from pathlib import Path

import holidays
import joblib
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# ============================================================
# 1. PATHS
# ============================================================

PROJECT = Path("/home/nia/Downloads/parkwise")

TRAFFIC_LOG_PATH = PROJECT / "parkwise_final_maybe/nairobi_parking_traffic_log.csv"
SPOTCHECK_PATH = PROJECT / "parkwise_final_maybe/nairobi_parking_spotcheck.csv"
MASTER_DATASET_PATH = (
        PROJECT / "parkwise_final_maybe/nairobi_parking_master_dataset.csv"
)
EXPANDED_OBS_PATH = PROJECT / "nairobi_parking_expanded_observations.csv"

ARTIFACT_DIR = PROJECT / "model2_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

MODEL_SAVE_PATH = ARTIFACT_DIR / "parkwise_model2_gbr.joblib"
FEATURE_SAVE_PATH = ARTIFACT_DIR / "model2_feature_list.json"

print("=" * 70)
print("PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR")
print("=" * 70)

# ============================================================
# 2. LOAD OBSERVATIONS
# ============================================================

if SPOTCHECK_PATH.exists() and len(pd.read_csv(SPOTCHECK_PATH)) >= 50:
    obs_df = pd.read_csv(SPOTCHECK_PATH)
    print(f"✓ Using spotcheck dataset: {len(obs_df)} rows")
elif EXPANDED_OBS_PATH.exists():
    obs_df = pd.read_csv(EXPANDED_OBS_PATH)
    print(f"✓ Using expanded observations: {len(obs_df)} rows")
else:
    raise FileNotFoundError("No observation dataset available.")

# Standardize columns
if (
        "parking_pressure_score" in obs_df.columns
        and "observed_occupancy_fraction" not in obs_df.columns
):
    obs_df.rename(
        columns={"parking_pressure_score": "observed_occupancy_fraction"},
        inplace=True,
    )

if "collected_at" in obs_df.columns and "timestamp" not in obs_df.columns:
    obs_df.rename(columns={"collected_at": "timestamp"}, inplace=True)

if "observed_at" in obs_df.columns and "timestamp" not in obs_df.columns:
    obs_df.rename(columns={"observed_at": "timestamp"}, inplace=True)

obs_df["timestamp"] = pd.to_datetime(obs_df["timestamp"], errors="coerce")
obs_df = (
    obs_df.dropna(subset=["timestamp"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

# Target: 0 to 100 percentage score
obs_df["parking_pressure_score"] = (
        obs_df["observed_occupancy_fraction"].astype(float).clip(0, 1) * 100
)

# Categorical Facility Names
if "facility_name" not in obs_df.columns:
    obs_df["facility_name"] = "CBD_Default"

# ============================================================
# 3. MERGE FACILITY METADATA & TRAFFIC
# ============================================================

merged_df = obs_df.copy()

if MASTER_DATASET_PATH.exists():
    master_df = pd.read_csv(MASTER_DATASET_PATH)
    meta_cols = [
        c
        for c in ["facility_name", "zone", "base_rate_kes", "capacity"]
        if c in master_df.columns
    ]
    if "facility_name" in meta_cols:
        master_subset = master_df[meta_cols].drop_duplicates(
            subset=["facility_name"]
        )
        merged_df = merged_df.merge(master_subset, on="facility_name", how="left")
        print("✓ Facility metadata merged.")

# Time extractors
merged_df["date"] = merged_df["timestamp"].dt.date
merged_df["hour"] = merged_df["timestamp"].dt.hour
merged_df["dayofweek"] = merged_df["timestamp"].dt.dayofweek
merged_df["month"] = merged_df["timestamp"].dt.month

# Traffic Merge (Fallback to hourly pattern if single-date logs present)
if TRAFFIC_LOG_PATH.exists():
    t_df = pd.read_csv(TRAFFIC_LOG_PATH)
    if "collected_at" in t_df.columns:
        t_df.rename(columns={"collected_at": "timestamp"}, inplace=True)
    t_df["timestamp"] = pd.to_datetime(t_df["timestamp"], errors="coerce")
    t_df["date"] = t_df["timestamp"].dt.date
    t_df["hour"] = t_df["timestamp"].dt.hour

    if "traffic_delay_index" in t_df.columns:
        if t_df["date"].nunique() > 1:
            t_profile = (
                t_df.groupby(["date", "hour"])["traffic_delay_index"]
                .mean()
                .reset_index()
            )
            merged_df = merged_df.merge(t_profile, on=["date", "hour"], how="left")
        else:
            t_profile = (
                t_df.groupby("hour")["traffic_delay_index"].mean().reset_index()
            )
            merged_df = merged_df.merge(t_profile, on="hour", how="left")
            print("⚠️ Traffic log has 1 date: Using mean hourly profile fallback.")

# ============================================================
# 4. ROBUST LAG FEATURE CREATION WITH TIME GAUGING
# ============================================================

print("\n" + "=" * 70)
print("BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS")
print("=" * 70)

merged_df = merged_df.sort_values(["facility_name", "timestamp"]).reset_index(
    drop=True
)

merged_df["time_diff_hours"] = (
        merged_df.groupby("facility_name")["timestamp"].diff().dt.total_seconds()
        / 3600.0
)

facility_group = merged_df.groupby("facility_name")["parking_pressure_score"]
merged_df["raw_prev_occ"] = facility_group.shift(1)

merged_df["previous_occupancy"] = np.where(
    merged_df["time_diff_hours"] <= 3.0, merged_df["raw_prev_occ"], np.nan
)

merged_df["occupancy_rolling_3"] = merged_df.groupby("facility_name")[
    "previous_occupancy"
].transform(lambda x: x.rolling(3, min_periods=1).mean())

facility_medians = merged_df.groupby("facility_name")[
    "parking_pressure_score"
].transform("median")
merged_df["previous_occupancy"] = merged_df["previous_occupancy"].fillna(
    facility_medians
)
merged_df["occupancy_rolling_3"] = merged_df["occupancy_rolling_3"].fillna(
    facility_medians
)

# ============================================================
# 5. CYCLIC AND CALENDAR FEATURES
# ============================================================

hour = merged_df["hour"]
day = merged_df["dayofweek"]

merged_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
merged_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
merged_df["day_sin"] = np.sin(2 * np.pi * day / 7)
merged_df["day_cos"] = np.cos(2 * np.pi * day / 7)
merged_df["is_weekend"] = day.isin([5, 6]).astype(int)
merged_df["is_peak_hour"] = hour.isin([7, 8, 9, 16, 17, 18]).astype(int)

ke_holidays = holidays.Kenya()
merged_df["is_public_holiday"] = merged_df["date"].apply(
    lambda d: int(d in ke_holidays)
)

merged_df["facility_name"] = merged_df["facility_name"].astype("category")

candidate_features = [
    "facility_name",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_weekend",
    "is_peak_hour",
    "is_public_holiday",
    "previous_occupancy",
    "occupancy_rolling_3",
]

if "traffic_delay_index" in merged_df.columns:
    merged_df["traffic_delay_index"] = (
        pd.to_numeric(merged_df["traffic_delay_index"], errors="coerce").fillna(
            merged_df["traffic_delay_index"].median()
        )
    )
    candidate_features.append("traffic_delay_index")

if "base_rate_kes" in merged_df.columns:
    merged_df["base_rate_kes"] = (
        pd.to_numeric(merged_df["base_rate_kes"], errors="coerce").fillna(
            merged_df["base_rate_kes"].median()
        )
    )
    candidate_features.append("base_rate_kes")

# ============================================================
# 6. CHRONOLOGICAL SPLIT (80/20)
# ============================================================

merged_df = merged_df.sort_values("timestamp").reset_index(drop=True)
split_idx = int(len(merged_df) * 0.80)

train_df = merged_df.iloc[:split_idx].copy()
test_df = merged_df.iloc[split_idx:].copy()

active_features = []
categorical_features = []

for c in candidate_features:
    if c == "facility_name":
        active_features.append(c)
        categorical_features.append(c)
        continue

    if train_df[c].nunique(dropna=True) >= 2:
        active_features.append(c)
    else:
        print(
            f"⚠️ Dropped '{c}': Low training variance ({train_df[c].nunique()} unique"
            " value)."
        )

X_train = train_df[active_features]
X_test = test_df[active_features]
y_train = train_df["parking_pressure_score"].astype(float)
y_test = test_df["parking_pressure_score"].astype(float)

print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)
print(f"Train Rows: {len(X_train)} | Test Rows: {len(X_test)}")
print(f"Active Features ({len(active_features)}): {active_features}")

# ============================================================
# 7. HISTORICAL BASELINE COMPARISON
# ============================================================

historical_avg = (
    train_df.groupby(["facility_name", "hour", "dayofweek"], observed=False)[
        "parking_pressure_score"
    ]
    .mean()
    .reset_index()
    .rename(columns={"parking_pressure_score": "baseline_prediction"})
)

test_baseline = test_df.merge(
    historical_avg, on=["facility_name", "hour", "dayofweek"], how="left"
)
test_baseline["baseline_prediction"] = test_baseline[
    "baseline_prediction"
].fillna(y_train.mean())

baseline_preds = test_baseline["baseline_prediction"].to_numpy()
baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))

print("\n" + "=" * 70)
print("HISTORICAL BASELINE PERFORMANCE")
print("=" * 70)
print(f"Baseline MAE:  {baseline_mae:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")

# ============================================================
# 8. TRAIN MODEL 2 (HYPERPARAMETER TUNING VIA RANDOMIZEDSEARCHCV)
# ============================================================

param_distributions = {
    "max_iter": randint(150, 450),
    "learning_rate": uniform(0.01, 0.08),
    "max_depth": randint(3, 8),
    "min_samples_leaf": randint(15, 60),
    "l2_regularization": uniform(0.5, 5.0),
    "max_leaf_nodes": randint(15, 63),
}

base_model = HistGradientBoostingRegressor(
    categorical_features=categorical_features, random_state=42
)

tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=25,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

print("\n" + "=" * 70)
print("EXECUTING RANDOMIZED SEARCH CV ACROSS TIME SERIES SPLITS")
print("=" * 70)

search.fit(X_train, y_train)

m2_model = search.best_estimator_

print("\n✓ Tuning Complete!")
print(f"Best CV Mean MAE: {-search.best_score_:.3f}")
print("Best Hyperparameters:")
for param, val in search.best_params_.items():
    print(f"  • {param}: {val}")

# ============================================================
# 9. EVALUATION
# ============================================================

m2_preds = np.clip(m2_model.predict(X_test), 0, 100)
m2_mae = mean_absolute_error(y_test, m2_preds)
m2_rmse = np.sqrt(mean_squared_error(y_test, m2_preds))
mae_improvement = baseline_mae - m2_mae

print("\n" + "=" * 70)
print("MODEL 2: GRADIENT BOOSTING RESULTS")
print("=" * 70)
print(f"Model 2 MAE:  {m2_mae:.3f}")
print(f"Model 2 RMSE: {m2_rmse:.3f}")

if m2_mae < baseline_mae:
    imp_pct = (mae_improvement / baseline_mae) * 100
    print(
        f"✓ MAE Improvement over Baseline: {mae_improvement:.3f} points"
        f" ({imp_pct:.2f}%)"
    )
else:
    print(
        f"⚠️ Model 2 MAE is {abs(mae_improvement):.3f} points worse than"
        " baseline."
    )

# ============================================================
# 10. PERMUTATION IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("PERMUTATION IMPORTANCE")
print("=" * 70)

perm = permutation_importance(
    m2_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1,
)

importance = pd.Series(
    perm.importances_mean, index=active_features
).sort_values(ascending=False)
for feat, imp in importance.items():
    print(f"{feat:<25} {imp:>10.5f}")

# ============================================================
# 11. SAVE ARTIFACTS
# ============================================================

joblib.dump(m2_model, MODEL_SAVE_PATH)
with open(FEATURE_SAVE_PATH, "w") as f:
    json.dump(
        {"features": active_features, "target": "parking_pressure_score"},
        f,
        indent=4,
    )

print("\n" + "=" * 70)
print(f"✓ Model saved: {MODEL_SAVE_PATH}")
print(f"✓ Features saved: {FEATURE_SAVE_PATH}")
print("=" * 70)

PARKWISE MODEL 2: OPTIMIZED PARKING PRESSURE PREDICTOR
✓ Using expanded observations: 36621 rows
✓ Facility metadata merged.
⚠️ Traffic log has 1 date: Using mean hourly profile fallback.

BUILDING TIME-AWARE FACILITY OCCUPANCY LAGS
⚠️ Dropped 'is_public_holiday': Low training variance (1 unique value).
⚠️ Dropped 'traffic_delay_index': Low training variance (1 unique value).

DATA SPLIT
Train Rows: 29296 | Test Rows: 7325
Active Features (10): ['facility_name', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend', 'is_peak_hour', 'previous_occupancy', 'occupancy_rolling_3', 'base_rate_kes']

HISTORICAL BASELINE PERFORMANCE
Baseline MAE:  10.814
Baseline RMSE: 13.701

EXECUTING RANDOMIZED SEARCH CV ACROSS TIME SERIES SPLITS
Fitting 5 folds for each of 25 candidates, totalling 125 fits

✓ Tuning Complete!
Best CV Mean MAE: 8.639
Best Hyperparameters:
  • l2_regularization: 2.729163764267956
  • learning_rate: 0.01799799326544023
  • max_depth: 5
  • max_iter: 237
  • max_leaf_node